<h6><center>Big Data : Enjeux, stockage et extraction</center></h6>

<h1>
<hr style=" border:none; height:3px;">
<center>TP n°2 : Introduction à Spark</center>
<hr style=" border:none; height:3px;">
</h1>

**Source** :
* Cours CentraleSupélec de Francesca Bugiotti
* Cours B.U.T Science des données de Alain Gely

In [ ]:
import multiprocessing

multiprocessing.cpu_count()

# 1. Introduction

Spark prend en charge quatre langages de programmation : Scala (langage natif), Java, R et Python. Dans ce TP, nous utiliserons l'API Python de Spark : **PySpark**. La section 2 présente quelques notions de programmation Spark à l'aide d'exemples simples. Dans la section 3, les exercices vous donneront l'occasion de mettre en pratique ces notions.

**Spark** (ou **Apache Spark**) est un framework open source de calcul distribué. Spark permet au développeur de se concentrer sur les problématiques métiers en masquant toute la complexité liée à l'exécution d'applications dans un environnement distribué : parallélisation du calcul, communication réseau et tolérance aux pannes.

Une application Spark s'exécute sous la forme d'un ensemble de processus indépendants (appelés *Executors*) distribués sur les noeuds (ou *Worker* Nodes) d'un cluster, et coordonnés par le *Driver*. Le _Driver_ crée un objet appelé **_SparkContext_** qui communique avec le gestionnaire de cluster sous-jacent et coordonne la distribution du calcul entre les _Executors_. Spark peut aussi s'exécuter sur des « clusters » locaux constitués d'une seule machine.

# 2. Présentation de Spark


In [ ]:
from pyspark import SparkContext, SparkConf
import os

In [ ]:
seed_value = 0
os.environ['PYTHONHASHSEED'] = str(seed_value)
# source n°1 : https://stackoverflow.com/questions/36798833/what-does-exception-randomness-of-hash-of-string-should-be-disabled-via-pythonh
# source n°2 : https://chenna.me/blog/2023/12/25/python-hash-is-not-deterministic/

In [ ]:
with open('section_2_exemple_1.txt', 'w') as fichier:
  fichier.write("- Bonjour, dit Alice\n- Bonjour, répondit Bob\n- Belle journée, Bob, dit Alice")

with open('section_2_exemple_2.txt', 'w') as fichier:
  fichier.write("- J'ai aimé discuté avec vous, dit Alice\n- Moi de même, dit Bob\n- Aurevoir, dit Alice\n- Aurevoir, répondit Bob")

## 2.1 Connexion à Spark

Un objet **SparkContext** représente une connexion à un cluster Spark. Il s'agit du point d'entrée pour accéder aux fonctionnalités de Spark.

In [ ]:
## Configuration

monAppli = "TPintroductionSpark" # nom de l'application

monCluster = "local[*]" # local : pas de gestionnaire de cluster (utilisation locale)
                        # [*] : nombre de cœurs à utiliser, * = tous les coeurs

conf = SparkConf().setAppName(monAppli).setMaster(monCluster)

## Instanciation

sc = SparkContext.getOrCreate(conf = conf) # getOrCreate() : créer ou réutiliser un SparkContext

print("Initialisation réussie !")

# sc.getConf().getAll()

In [ ]:
#sc.stop()

In [ ]:
sc.startTime

In [ ]:
sc

## 2.2 Création d'un RDD (Resilient Distributed Dataset)

Spark distribue les données et les calculs entre les machines d'un cluster en utilisant la notion de **Resilient Distributed Dataset (RDD)**. Un RDD est une collection **immuable** de données distribuées pouvant être traitées en parallèle.

Le _SparkContext_ divise un RDD en plusieurs _partitions_ et les répartit sur différentes machines du cluster. Une application Spark crée des RDD et spécifie les calculs à effectuer sur ces RDD, en utilisant les fonctions spéciales fournies à cet effet par Spark.

In [ ]:
print(f"Par défaut, un RDD est divisé en {sc.defaultMinPartitions} partitions minimum.")

Spark propose deux méthodes pour créer un RDD :
- Distribuer une collection d'objets locale
- Charger un ensemble de données externes, stockées dans un système de fichier local ou distribué ou dans une base de données

### 2.2.1 Collection d'objets locale

Fonction Spark :
* $sc.parallelize()$ : Distribuer une collection Python locale pour former un RDD.

In [ ]:
mesDonnees = [1, 2, 3, 4, 5, 6, 7] # collection Python locale

# Distribuer une collection d'objets locale

monRDD = sc.parallelize(mesDonnees)
print(f"Le RDD créé a {monRDD.getNumPartitions()} partitions")

# Indiquer le nombre minimal de partitions

monRDD = sc.parallelize(mesDonnees, 3)
print(f"Le RDD créé a {monRDD.getNumPartitions()} partitions")

# Récupérer localement le contenu du RDD

print("RDD :", monRDD.collect())

In [ ]:
# Autre exemple

monRDD = sc.range(0, 10, 2) # 0 à 10 (exclu) par pas de 2
monRDD = sc.range(0, 10, 2, 3) # .. en spécifiant 3 partitions
print(monRDD.collect())

### 2.2.2 Un ou plusieurs fichiers

Les fichiers doivent être accessibles à tous les nœuds (partage réseau, HDFS, copie locale, …).

Fonctions Spark :
* $sc.textFile()$ : Lire **un ou plusieurs fichiers** (à séparer par des virgules). Les fichiers sont segmentés en lignes.
* $sc.wholeTextFiles()$ : Lire **tous les fichiers d'un dossier**. Chaque fichier est lu comme un enregistrement unique et renvoyé sous forme de paire clé-valeur, où la clé correspond au chemin d'accès de chaque fichier et la valeur à son contenu.

In [ ]:
# Lire un fichier

monRDD = sc.textFile("moby-dick.txt")

# Spécifier 3 partitions au minimum

monRDD = sc.textFile("moby-dick.txt",3)

print(f"Le RDD contient {monRDD.count()} enregistrements.")
print()

# Récupérer localement les 10 première lignes du RDD

monRDD.take(10)

In [ ]:
# Lire plusieurs fichiers simultanément

monRDD = sc.textFile("section_2_exemple_1.txt,section_2_exemple_2.txt")

print(f"Le RDD contient {monRDD.count()} enregistrements.")
print("Contenu du RDD :")
print(monRDD.collect())
print()

# Utiliser les jokers (*)

monRDD = sc.textFile("./sample_data/*.csv")

print(f"Le RDD contient {monRDD.count()} enregistrements.")
print("Premières lignes du RDD :")
monRDD.take(2)

In [ ]:
# Lire tous les fichiers d'un dossier

monRDD = sc.wholeTextFiles("./*_exemple_*.txt")

print(f"Le RDD contient {monRDD.count()} enregistrements.")
monRDD.collect() # notez le format paires clé-valeur en sortie

## 2.3 Opérations sur un RDD

Une fois créé, Spark propose deux catégories d'opérations sur un RDD :

* **Transformations**. Une transformation prend en entrée un ou plusieurs RDD et renvoie un nouveau RDD.  
* **Actions**. Une action prend en entrée un RDD et renvoie une valeur.

Une application Spark est une séquence d'opérations qui créent, transforment et effectuent certaines actions sur des RDD.

Les *transformations* ne sont pas exécutées par Spark tant qu'aucune *action* n'est invoquée. C'est ce qu'on appelle l'**évaluation paresseuse**. Les fonctions de création d'un RDD sont elles aussi "paresseuses", ce qui évite de charger en mémoire l'intégralité du contenu d'un fichier (qui peut être très volumineux). L'évaluation paresseuse permet à Spark d'optimiser les opérations avant exécution.

### 2.3.1 Transformations

Voici quelques transformations courantes en Spark. Dans la liste suivante,  $rdd$  désigne le RDD auquel la transformation est appliquée.

* $rdd.filter(func)$. Renvoie un RDD composé uniquement des éléments du RDD d'entrée $rdd$ qui satisfont la condition décrite par $func$.
* $rdd.map(func)$. Applique une fonction $func$ à chaque élément du RDD d'entrée $rdd$ et renvoie un RDD contenant le résultat.
* $rdd.flatMap(func)$. Identique à $map()$, mais si le résultat est une liste ou un tuple, $flatmap$ va transformer chaque élément de la liste ou du tuple en un enregistrement distinct du RDD final.
* $rdd.union(other)$. Prend deux RDDs ($rdd$ et $other$) et renvoie un RDD qui contient les éléments des deux. Contrairement à l'opération mathématique, $union$ ne supprime pas les doublons.
* $rdd.intersection(other)$. Prend deux RDDs ($rdd$ et $other$) et renvoie un RDD qui contient les éléments communs aux deux.
* $rdd.subtract(other)$. Prend deux RDD ($rdd$ et $other$) et renvoie un RDD contenant les éléments du RDD $rdd$, à l'exception de ceux qui se trouvent dans $other$.
* $rdd.cartesian(other)$. Prend deux RDD ($rdd$ et $other$) et renvoie un RDD qui contient le produit cartésien des deux.
* $rdd.distinct()$. Renvoie un RDD contenant les mêmes éléments que le RDD d'entrée $rdd$ sans doublon.

#### Exemple $map()$


Voici un exemple d'utilisation de la transformation $map()$.

Les fonctions $filter()$, $map()$ et $flatMap()$ prennent en entrée une fonction. Une manière élégante de passer une fonction à une fonction en Python est d'utiliser les **fonctions anonymes** (lambda fonction).

Dans le code ci-dessous, l'argument de la fonction $map()$ est une fonction qui prend en entrée une variable $x$. Le type de cette variable doit correspondre au type des éléments composant le RDD $rdd$, c'est-à-dire ici des entiers. La fonction renvoie le carré de $x$.

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4])
rdd_square = rdd.map(lambda x : x*x) # du RDD vers un RDD

print("`map()` :", rdd_square.collect())

De façon équivalent, vous pouvez définir explicitement la fonction plutôt que de passer par la notation lambda, mais le code s'en trouve alourdi :

In [ ]:
def square(x):
  return x*x

rdd_square = rdd.map(square)

print("`map()` :", rdd_square.collect())

Préférez utiliser une fonction anonyme quand l'opération à appliquer est simple.

#### Exemple $flatMap()$


Voici un exemple d'utilisation de la transformation $flatMap()$.

$flatMap()$ commence par appliquer une fonction à chaque élément du RDD, tout comme $map()$, puis, si le résultat est une liste ou un tuple, "aplatit" cette liste ou ce tuple. Autrement dit, lorsque $map()$ renvoie un RDD dont chaque élément est une liste ou un tuple, $flatMap()$ renvoie un RDD dont chaque élément est une valeur de cette liste ou de ce tuple.

In [ ]:
rdd = sc.parallelize(['abc','de'])

rdd_map = rdd.map(list)
rdd_flatmap = rdd.flatMap(list)

print("`map()` :", rdd_map.collect())
print("`flatMap()` :", rdd_flatmap.collect())

#### Exemple $filter()$

Voici un exemple d'utilisation de la transformation $filter()$.

In [ ]:
names = sc.parallelize(["Anne", "Jacques", "Ali", "Léa", "Marc", "Nathan", "Nick", "Marie", "Mohammed"])

rdd_filter = names.filter(lambda x: (x[0] == "M"))

print("`filter()` :", rdd_filter.collect())

#### Exemple de $union()$, $intersection()$, $subtract()$, $cartesian()$

Voici un exemple d'utilisation des transformations $union()$, $intersection()$, $subtract()$ et $cartesian()$.

In [ ]:
rdd1 = sc.parallelize([1, 2, 3, 4])
rdd2 = sc.parallelize([3, 4, 5, 6, 7])

union = rdd1.union(rdd2)
intersection = rdd1.intersection(rdd2)
subtract = rdd1.subtract(rdd2)
cartesian = rdd1.cartesian(rdd2)

print("Contenu du RDD rdd1 :", rdd1.collect())
print("Contenu du RDD rdd2 :", rdd2.collect())
print("`union()` :", union.collect()) # notez que les données ne sont pas dédupliquées
print("`intersection()` :", intersection.collect())
print("`subtract()` :", subtract.collect())
print("`cartesian()` :", cartesian.collect())

#### Exemple $distinct()$

Voici un exemple d'utilisation de la transformation $distinct()$, qui s'utilise sans argument (excepté éventuellement le nombre de partitions).

In [ ]:
no_duplicates = union.distinct()

print("Contenu du RDD union: ", union.collect())
print("Contenu du RDD union (sans doublon): ", no_duplicates.collect())

In [ ]:
union.distinct().toDebugString()

### 2.3.2 Actions

Voici quelques actions courantes en Spark. Dans la liste suivante,  $rdd$  désigne le RDD auquel la transformation est appliquée.  
* $rdd.reduce(func)$. Agrège deux à deux les éléments du RDD d'entrée selon la fonction $func$ passée en argument. La fonction doit être commutative et associative.
* $rdd.collect()$. Renvoie une liste Python contenant tous les éléments du RDD d'entrée.
* $rdd.count()$. Renvoie le nombre d'éléments dans le RDD d'entrée.
* $rdd.countByValues()$. Renvoie le nombre d'occurrences de chaque élément dans le RDD d'entrée.
* $rdd.take(num)$. Affiche les $num$ premiers éléments du RDD d'entrée.
* $rdd.top(num)$. Affiche les $num$ premiers éléments du RDD d'entrée trié par ordre décroissant.

#### Exemple $reduce()$.

Puisque l'action `reduce()` agrège deux à deux les éléments du RDD auquel elle est appliquée, la fonction transmise à `reduce()` doit prendre en entrée **deux arguments**, du même type que les éléments du RDD.

In [ ]:
rdd = sc.parallelize([1, 2, 3, 4])

rdd_sum = rdd.reduce(lambda x, y : x + y) # du RDD vers une valeur

print("Contenu du RDD rdd :", rdd.collect())
print("`reduce()` :", rdd_sum)

#### Exemple $countByValue()$

Le résultat de `countByValue()` est un dictionnaire Python, où chaque clé est un élément du RDD d'entrée et où la valeur associée correspond au nombre d'occurrence de la clé.

In [ ]:
rdd_occurrences = union.countByValue()

print("Contenu du RDD union :", union.collect())
print("Occurrence de chaque élément du RDD union :", rdd_occurrences)

## 2.4 Paired RDD

Les **Paired RDDs** sont des RDDs dont les éléments sont des tuples / listes contenant deux éléments qui forment des **paires clé-valeur** : le premier élément est la clé, le deuxième est la valeur.

Le *paradigme map / reduce* peut être appliqué à des Paired RDD via les fonctions Spark s'appliquant aux clés ou aux valeurs. Les Paired RDD sont souvent obtenus à la suite d'une transformation $map()$.

### 2.4.1 Transformations

Toutes les transformations appliquées aux RDDs standards peuvent également être appliquées aux Paired RDD. Cependant, les fonctions passées en argument doivent prendre en entrée des paires plutôt que des valeurs uniques.

Certaines transformations sont quant à elle spécifiques aux Paired RDDs :

* $rdd.reduceByKey(func)$. Agrège les valeurs par clé à l'aide d'une fonction $func$ associative et commutative. Similaire à l'étape *Combine* de *MapReduce*.
* $rdd.sortBy(func, ascending=...)$. Trie le RDD d'entrée selon les clés calculées par la fonction $func$ et renvoie un RDD contenant le résultat du tri.
* $rdd.groupByKey()$. Regroupe les valeurs du RDD par clé. Le résultat est un itérable.
* $rdd.join(other)$. Joint les valeurs des RDDs $rdd$ et $other$ dont les clés correspondent.
* $rdd.mapValues()$. Applique une fonction $func$ à chaque élément du RDD d'entrée sans modifier les clés et renvoie un RDD contenant le résultat.

In [ ]:
sample_rdd = sc.parallelize(['do', 'you', 'enjoy', 'this', 'exercise', '?', 'yes', 'I', 'hope'])

'''
Transformer `sample_rdd` en un RDD où chaque élément est (len(s), s), c'est-à-dire un Paired RDD.
Dans l'instruction suivante, x représente un élément du RDD d'entrée, soit une chaîne de caractère.
'''
sample_rdd = sample_rdd.map(lambda x: (len(x), x))
print("Paires clé-valeur (`map()`) :", sample_rdd.collect())
print()

'''
Transformer le RDD précédent en un RDD dans lequel les éléments sont triés par clé dans l'ordre décroissant.
Dans l'instruction suivante, x représente un élément du RDD d'entrée, soit un tuple (len(s), s)).
x[0] désigne le premier élément du tuple, c'est-à-dire len(s).
'''
sample_rdd = sample_rdd.sortBy(lambda x: x[0], ascending=False)
print("Paires clé-valeur ordonnées (`sortBy()`) :", sample_rdd.collect())
print()

'''
Regrouper les éléments par clé à l'aide de `groupByKey()`.
Le résultat est un RDD, où chaque élément est un tuple (k, L),
k étant un entier et L un itérable (un objet qui permet d'itérer sur une liste).
'''
sample_rdd_gbk = sample_rdd.groupByKey()
print("1. Paires clé-valeur groupées (`groupByKey()`) :", sample_rdd_gbk.collect())
print()

'''
Convertir l'itérable en liste grâce à `mapValues()`
'''
sample_rdd_gbk = sample_rdd_gbk.mapValues(list)
print("2. Paires clé-valeur groupées (`groupByKey()` & `mapValues()`) :", sample_rdd_gbk.collect())
print()

'''
Il est également possible de regrouper les éléments par clé à l'aide des deux instructions suivantes.
Le résultat est un RDD, où chaque élément est (k, L),
k étant un entier et L une liste (et non un itérable comme précédemment).
'''
# L'instruction suivante transforme chaque tuple (len(s), s) du RDD d'entrée en un tuple (len(s), [s]).
# Le deuxième élément du tuple devient donc une liste.
sample_rdd_gbk_alternative = sample_rdd.map(lambda x: (x[0], [x[1]]))

# Étant donné deux tuples du RDD d'entrée, (len(s1), [s1]) et (len(s2), [s2]),
# x et y dans l'instruction suivante sont respectivement [s1] et [s2].
# L'instruction x + y concatène les deux listes [s1] et [s2], ce qui donne la liste [s1, s2].
sample_rdd_gbk_alternative = sample_rdd_gbk_alternative.reduceByKey(lambda x, y: x+y)
print("3. Paires clé-valeur groupées (`reduceByKey()`) :", sample_rdd_gbk_alternative.collect())
print()

'''
Ne conserver que les mots de plus de 3 lettres.
Notez qu'on utilise ici l'une des transformations applicables aux RDDs standards.
'''
sample_rdd_filter = sample_rdd_gbk_alternative.filter(lambda x: (x[0] > 3))
print("Paires clé-valeur filtrées (`filter()`) :", sample_rdd_filter.collect())
print()

'''
Ne renvoyer que les mots.
Notez qu'on utilise ici l'une des transformations applicables aux RDDs standards.
'''
sample_rdd_reduce = sample_rdd_filter.flatMap(lambda x: x[1])
print("Liste de mots de plus de 3 lettres (`flatMap()`) :", sample_rdd_reduce.collect())

Le code suivant montre une manière élégante d'écrire une séquence de transformations. Les transformations sont enchaînées les unes après les autres. Chaque ligne contient une transformation. Le symbole « \ » indique que l'instruction Python se poursuit à la ligne suivante. Utilisez ce style dans les exercices qui vous seront proposés plus tard.

In [ ]:
sample_rdd = sc.parallelize(['do', 'you', 'enjoy', 'this', 'exercise', '?', 'yes', 'I', 'hope'])\
               .map(lambda x: (len(x), x))\
               .map(lambda x: (x[0], [x[1]]))\
               .reduceByKey(lambda x, y: x+y)\
               .sortBy(lambda x: x[0], ascending=False)

sample_rdd.collect()

In [ ]:
# Exemple avec `join()`

rdd1 = sc.parallelize([('first', 1), ('second', 2), ('third', 3)])
rdd2 = sc.parallelize([('first', 2), ('second', 4), ('third', 9)])

rdd_join = rdd1.join(rdd2)
rdd_join.collect()

### 2.4.2 Actions

Comme pour les transformations, toutes les actions disponibles pour les RDDs standards peuvent également être utilisées sur les Paired RDD.

Certaines actions sont quant à elle spécifiques aux Paired RDDs :

* $countByKey()$ : Renvoie un dictionnaire Python, où chaque clé est mappée au nombre de valeurs associées à cette clé.
* $collectAsMap()$ : Renvoie le RDD sous forme de dictionnaire (tandis que la fonction $collect()$ renvoie le RDD standard sous forme de liste).
* $lookup(key)$ : Renvoie une liste contenant toutes les valeurs associées à la clé $key$ donnée.

# 3. Exercices

In [ ]:
import random

## 3.1 Exercice 1

In [ ]:
# Tirer 50 nombres aléatoires entre 0 et 100
T = [random.randint(0,100) for i in range(50)]
print(T)

1. Écrire une application Spark calculant la somme des valeurs distinctes d'un RDD contenant des nombres tirés au hasard.

In [ ]:
somme = sc.parallelize(T)\
          .distinct()\
          .reduce(lambda x, y: x + y) # ou sum() https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.sum.html

print(f"La somme vaut {somme}")

2. Écrire, selon le **paradigme MapReduce**, une application Spark calculant la moyenne des valeurs distinctes d'un RDD contenant des nombres tirés au hasard. Remarque : la dernière opération est un calcul hors Spark.

In [ ]:
res = sc.parallelize(T)\
        .map(lambda num: (num,1))\
        .reduce(lambda paire1, paire2 : (paire1[0] + paire2[0], paire1[1] + paire2[1]))

moyenne = res[0]/res[1]

print(f"La moyenne vaut {moyenne}")

In [ ]:
# OU proposé par un étudiant, plus proche du paradigme "MapReduce"

rdd_moyenne = sc.parallelize(T)\
                .map(lambda num: ("stat",(num,1)))\
                .reduceByKey(lambda paire1, paire2 : (paire1[0] + paire2[0], paire1[1] + paire2[1]))

res = rdd_moyenne.collect()
somme, count = res[0][1]
moyenne = somme / count

print(moyenne)

## 3.2 Exercice 2

On s'intéresse au résultat du tiercé. On suit une cinquantaine de chevaux. Le jeu de données contient plusieurs tirages du tiercé sous la forme d'une liste de liste : chaque élément du jeu de données est une liste de 3 entiers, représentants les 3 chevaux arrivés en tête par ordre d'arrivée.

1. Générer au hasard une centaine de tiercés à l'aide de `random.sample()`.

In [ ]:
horses = range(0,50)
T = [random.sample(horses,3) for _ in range(100)]

2. Construire un RDD contenant la liste des chevaux arrivés en première position de chaque tiercé.

In [ ]:
rdd = sc.parallelize(T)\
        .map(lambda x: x[0])

rdd.take(3)

3. En utilisant `flatMap()` et `reduceByKey()`, calculer le nombre de fois où chaque cheval apparaît dans un tiercé. Afficher les 5 premiers éléments du RDD ainsi construit.

In [ ]:
rdd = sc.parallelize(T)\
        .flatMap(lambda x: [(x[0],1),(x[1],1),(x[2],1)])\
        .reduceByKey(lambda x, y: x + y)

rdd.take(5)

4. Modifier les fonctions passées en argument de `flatMap()` et `reduceByKey()` afin de déterminer la meilleure performance (premier, second ou troisième) réalisée par chacun des chevaux au cours des 100 courses. Ne conserver que les chevaux n'étant jamais arrivés premier à l'aide de `filter()`. Combien y a-t-il de chevaux concernés ?

In [ ]:
rdd = sc.parallelize(T)\
        .flatMap(lambda x: [(x[0],1),(x[1],2),(x[2],3)])\
        .reduceByKey(lambda x,y: min(x,y))\
        .filter(lambda x: x[1] > 1)

rdd.count()

5. Construire un RDD contenant la liste des performances d'un cheval. Par exemple `(8,[3,3,3,2,2,1,3])` si le cheval numéro 8 a été troisième sur 4 tiercés, second sur 2 tiercés et premier sur 1 tiercé.

In [ ]:
rdd = sc.parallelize(T)\
        .flatMap(lambda x: [(x[0],[1]),(x[1],[2]),(x[2],[3])])\
        .reduceByKey(lambda x, y: x + y)

rdd.take(3)

## 3.3 Exercice 3

In [ ]:
notes = sc.parallelize([("alice",12), ("bob",15), ("alice", 18), ("bob",3), ("charlène",15)])
info = sc.parallelize([("alice","SD"), ("bob", "SD"), ("charlène", "INFO")])

1. À l'aide de la tranformation `join()`, écrire une application Spark permettant d'obtenir le RDD ci-dessous et afficher le contenu du RDD.   
```
[ ('bob', (15, 'SD')),
  ('bob', (3, 'SD')),
  ('alice', (12, 'SD')),
  ('alice', (18, 'SD')),
  ('charlène', (15, 'INFO')) ]
```

In [ ]:
rdd = notes.join(info)

rdd.collect()

2. À partir du RDD précédent, produire le RDD suivant :
```
[ ('SD', 15) , ('SD', 3) , ('INFO', 15) , ('SD', 12) , ('SD', 18) ]
```

In [ ]:
rdd1 = rdd.map(lambda x: (x[1][1], x[1][0]))

rdd1.collect()

3. À partir du RDD précédent, calculer les moyennes par département en plusieurs étapes.

In [ ]:
rdd2 = rdd1.groupByKey()\
           .mapValues(lambda x: sum(x)/len(x))

rdd2.collect()

4. Il y a maintenant une coquille dans le RDD `info`, `charlène` est écrit sans accent. Vérifier que, dans ce cas, le département `INFO` n'apparaît pas dans le résultat de la jointure, et expliquer pourquoi. En utilisant un autre type de jointure, obtenir un RDD ou `charlène`, `charlene`, et le département `INFO` apparaissent.

Documentation Spark : [source](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.RDD.html). Utiliser CTRL + F pour trouver un mot dans une page.

In [ ]:
info = sc.parallelize([("alice","SD"), ("bob", "SD"), ("charlene", "INFO")])

In [ ]:
notes.join(info).collect()

Le département `INFO` n'apparaît pas dans le résultat de la jointure parce que la jointure se fait sur les clés (premier élément de chaque tuple), et qu'il n'y a pas de clé `charlene` dans le RDD `notes`.

In [ ]:
notes.fullOuterJoin(info).collect()

## 3.4 Exercice 4

In [ ]:
import re
import zipfile

In [ ]:
def nettoyage_texte(texte):
    # Remplacer les caractères qui ne sont ni des lettres ni des espaces par un espace
    etape1 = re.sub(r"[^a-zA-Z\s]" ," ", texte)
    # Remplacer plusieurs espaces successifs par un seul espace
    etape2 = re.sub(r"\s+", " ", etape1)
    return etape2.lower()

In [ ]:
with zipfile.ZipFile('conan_doyle.zip', 'r') as zip_ref:
    zip_ref.extractall('conan_doyle')

Créer un dossier `doyle` et y placer les textes de Conan Doyle.

1. Créer un RDD à partir du texte `La_Marque_des_quatre.txt` de Conan Doyle.

In [ ]:
doyle = sc.textFile("conan_doyle/La_Marque_des_quatre.txt")

2. Écrire une fonction permettant de compter le nombre de mots dans un fichier texte. Ne pas compter les caractères qui ne sont pas des lettres (ex : guillemets, virgules, nombres). Appliquer cette fonction au RDD de la question 1.

In [ ]:
def compter_mots(rdd):
  nb_mots = rdd.map(nettoyage_texte)\
               .flatMap(lambda ligne: ligne.split())\
               .map(lambda mot: 1)\
               .reduce(lambda x, y: x + y)
  return nb_mots

print(f'Il y a {compter_mots(doyle)} mots dans le texte "La Marque des Quatre"')

In [ ]:
# OU
doyle.map(nettoyage_texte)\
     .map(lambda ligne: len(ligne.split()))\
     .reduce(lambda x, y: x + y)

3. Reprendre la question 2 mais en passant plusieurs fichiers comme paramètre à la fonction `textFile()`.

In [ ]:
doyle = sc.textFile("conan_doyle/La_Marque_des_quatre.txt,conan_doyle/La_Résurrection_de_Sherlock_Holmes.txt")

print(f'Il y a {compter_mots(doyle)} mots dans le texte "La Marque des Quatre" et "La Résurrection de Sherlock Holmes"')

4. Reprendre la question 2, mais en utilisant `wholeTextFiles()` pour construire un RDD à partir de tous les fichiers textes d'un dossier.

In [ ]:
doyle = sc.wholeTextFiles("conan_doyle/*.txt")

doyle.map(lambda ligne: ligne[1])\
     .map(nettoyage_texte)\
     .flatMap(lambda ligne: ligne.split())\
     .map(lambda mot: 1)\
     .reduce(lambda x, y: x + y)

5. Calculer le nombre de mots **par fichier**.

In [ ]:
doyle.map(lambda x: (x[0], nettoyage_texte(x[1])))\
     .map(lambda x: (x[0], len(x[1].split())))\
     .collect()

6. Compter le nombre d'occurrences par mot et par fichier.

In [ ]:
doyle.map(lambda x: (x[0], nettoyage_texte(x[1])))\
     .map(lambda x: (x[0], x[1].split()))\
     .flatMap(lambda x: [((x[0], w) ,1) for w in x[1]])\
     .reduceByKey(lambda x, y: x + y)\
     .takeOrdered(20, lambda x: -x[1])

## 3.5 Exercice 5

In [ ]:
import random
import math

Le code ci-dessous permet de générer des commandes sous la forme de triplets : (identifiant de la commande, article commandé, prix de l'article commandé).

In [ ]:
def genereChaineCaractere():
    # Génération d'une chaîne de caractères aléatoire
    t = range(65,62+26)
    x = random.choices(t,k=8)
    l = [chr(r) for r in x]
    return ''.join(l)

def genereListeCommandes(n):
    commandes = []
    for i in range(n):
        x = random.random()
        nbArticles = int(1 + abs(math.log(x,2)))
        for j in range(nbArticles):
            commandes.append([f"C{i}", genereChaineCaractere(), int(1 + random.random()*100)]) # prix aléatoire
    return commandes

listeCommande = genereListeCommandes(3)

print(listeCommande)

1. Créer une liste de 1000 commandes et le RDD `rdd_commandes` correspondant.

In [ ]:
listeCommande = genereListeCommandes(1000)
rdd_commandes =  sc.parallelize(listeCommande)

2. Calculer le nombre moyen d'articles par commande. Par exemple, pour le jeu de données ci-dessous, le nombre moyen d'articles par commande est $\frac{1 + 1 + 2}{3} = 1.33$.
```
[['C0', 'EAWVMJMQ', 15],
 ['C1', 'GKLSASHF', 91],
 ['C2', 'SFFTBQUD', 77],
 ['C2', 'CGSJDCKD', 69]]
 ```

In [ ]:
res = rdd_commandes.map(lambda article: (article[0], 1))\
                   .reduceByKey(lambda x, y: x + y)\
                   .map(lambda commande: (commande[1],1))\
                   .reduce(lambda c1, c2: (c1[0] + c2[0], c1[1] + c2[1]))

moyenne = res[0]/res[1]

print(f"En moyenne, une commande comprend {moyenne} articles")

3. Transformer le RDD initial en un Paired RDD, tel que la clé d'un élément soit le numéro de commande et la valeur la liste des articles associés à la commande.

In [ ]:
rdd = rdd_commandes.map(lambda x : (x[0], [x[1]]))\
                   .reduceByKey(lambda x, y: x + y)

rdd.take(10)

In [ ]:
# OU
rdd = rdd_commandes.map(lambda elt: (elt[0],elt[1]) )\
                   .groupByKey()\
                   .mapValues(list)
rdd.take(10)